
# 03 — Accuracy Validation

Deep-dive into forecast accuracy beyond a single MAPE number.

Sections:
1. Setup & data pipeline  
2. Time-series cross-validation (walk-forward)  
3. Residuals analysis  
4. Per-store MAPE breakdown  
5. Error distribution  
6. Confidence interval coverage  
7. Summary & recommendations


In [ ]:

# ── Setup ─────────────────────────────────────────────────────────────────

import os, sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.data_validation import validate_dataset
from src.preprocessing import clean_dataframe
from src.feature_engineering import create_all_features
from src import forecasting

# ── Rebuild the full feature-engineered dataset ───────────────────────────

train = pd.read_csv("../data/raw/train.csv", parse_dates=["Date"])
store = pd.read_csv("../data/raw/store.csv")

df_raw = train.merge(store, on="Store", how="left").rename(columns={
    "Date": "date", "Sales": "sales_qty",
    "Store": "store_id", "Promo": "is_promotion",
})
keep = [c for c in ["date","sales_qty","store_id","is_promotion"] if c in df_raw.columns]

df_features = create_all_features(
    clean_dataframe(validate_dataset(df_raw[keep]))
)
print(f"Dataset ready: {df_features.shape}")

# ── Train both models (time-based split) ─────────────────────────────────

print("Training Prophet…")
prophet_model, prophet_metrics = forecasting.train_prophet(df_features)

print("Training XGBoost…")
xgb_model, xgb_metrics, feature_cols = forecasting.train_xgboost(df_features)

results = {"Prophet": prophet_metrics, "XGBoost": xgb_metrics}
best_name = forecasting.select_best_model(results)
print(f"\nBest model: {best_name}")



## 1. Walk-Forward Cross-Validation

Rolling out-of-sample evaluation using **3 folds** (each fold = 42 days). The training window expands with each fold — no data leakage.

```
Fold 1: train=[start … T-84],  test=[T-84 … T-42]
Fold 2: train=[start … T-42],  test=[T-42 … T]
Fold 3 (holdout): same as model training split above
```


In [ ]:

# ── Walk-Forward Cross-Validation ─────────────────────────────────────────
# Each fold uses a 42-day test window; the train window expands.

FOLD_SIZE = 42
N_FOLDS   = 3

df_sorted = df_features.sort_values("date").reset_index(drop=True)
max_date  = df_sorted["date"].max()

cv_results = []

for fold in range(N_FOLDS, 0, -1):
    test_end   = max_date - pd.Timedelta(days=FOLD_SIZE * (fold - 1))
    test_start = test_end - pd.Timedelta(days=FOLD_SIZE - 1)
    cutoff     = test_start - pd.Timedelta(days=1)

    train_fold = df_sorted[df_sorted["date"] <= cutoff].copy()
    test_fold  = df_sorted[
        (df_sorted["date"] >= test_start) & (df_sorted["date"] <= test_end)
    ].copy()

    if len(train_fold) < FOLD_SIZE * 2 or len(test_fold) == 0:
        continue

    fold_label = f"Fold {N_FOLDS - fold + 1}  ({test_start.date()} → {test_end.date()})"

    # --- XGBoost CV ---
    candidate_features = [
        "day_of_week","month","week_of_year","quarter",
        "is_weekend","is_month_start","is_month_end",
        "sales_lag_7","sales_lag_14","sales_lag_28",
        "rolling_mean_7","rolling_mean_28","rolling_std_7",
        "is_promotion",
    ]
    fold_feat_cols = [c for c in candidate_features if c in df_sorted.columns]

    train_clean = train_fold[["date","sales_qty"] + fold_feat_cols].dropna(subset=fold_feat_cols)
    test_clean  = test_fold[["date","sales_qty"] + fold_feat_cols].dropna(subset=fold_feat_cols)

    from xgboost import XGBRegressor
    xgb_cv = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0,
    )
    xgb_cv.fit(train_clean[fold_feat_cols], train_clean["sales_qty"])
    xgb_pred = xgb_cv.predict(test_clean[fold_feat_cols])
    xgb_m = forecasting.evaluate_forecast(test_clean["sales_qty"].values, xgb_pred)

    cv_results.append({
        "Fold": fold_label, "Model": "XGBoost",
        **xgb_m,
        "train_rows": len(train_clean), "test_rows": len(test_clean),
    })

cv_df = pd.DataFrame(cv_results)
print("Walk-Forward CV Results (XGBoost):")
display(cv_df[["Fold","Model","MAE","RMSE","MAPE","train_rows","test_rows"]]
        .style.format({"MAE":"{:.0f}","RMSE":"{:.0f}","MAPE":"{:.2f}%"}))

print(f"\nMean CV MAPE: {cv_df['MAPE'].mean():.2f}%  ±  {cv_df['MAPE'].std():.2f}%")



## 2. Residuals Analysis

Residuals = actual − predicted. A well-calibrated model should show residuals centred at zero with no systematic time-based pattern.


In [ ]:

# ── Compute residuals on the held-out test set (XGBoost) ──────────────────

SPLIT_DAYS = 42
df_sorted = df_features.sort_values("date").reset_index(drop=True)
cutoff     = df_sorted["date"].max() - pd.Timedelta(days=SPLIT_DAYS)

candidate_features = [
    "day_of_week","month","week_of_year","quarter",
    "is_weekend","is_month_start","is_month_end",
    "sales_lag_7","sales_lag_14","sales_lag_28",
    "rolling_mean_7","rolling_mean_28","rolling_std_7",
    "is_promotion",
]
feat_cols = [c for c in candidate_features if c in df_sorted.columns]

test_df = (
    df_sorted[df_sorted["date"] > cutoff][["date","sales_qty"] + feat_cols]
    .dropna(subset=feat_cols)
    .copy()
)

test_df["predicted"]  = np.clip(xgb_model.predict(test_df[feat_cols]), 0, None)
test_df["residual"]   = test_df["sales_qty"] - test_df["predicted"]
test_df["abs_pct_err"] = np.where(
    test_df["sales_qty"] != 0,
    np.abs(test_df["residual"] / test_df["sales_qty"]) * 100,
    np.nan,
)

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Residuals Over Time", "Actual vs Predicted"))

fig.add_trace(go.Scatter(
    x=test_df["date"], y=test_df["residual"],
    mode="markers", name="Residual",
    marker={"color": "steelblue", "size": 3, "opacity": 0.5},
), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

fig.add_trace(go.Scatter(
    x=test_df["sales_qty"], y=test_df["predicted"],
    mode="markers", name="Actual vs Predicted",
    marker={"color": "mediumseagreen", "size": 3, "opacity": 0.5},
), row=2, col=1)
# Perfect prediction line
max_val = max(test_df["sales_qty"].max(), test_df["predicted"].max())
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode="lines", name="Perfect", line={"color": "red", "dash": "dash"},
), row=2, col=1)

fig.update_layout(template="plotly_white", height=600, showlegend=False,
                  title_text="XGBoost Residuals — Test Set")
fig.show()

print(f"Mean residual : {test_df['residual'].mean():.2f}  (bias)")
print(f"Std  residual : {test_df['residual'].std():.2f}")
print(f"Mean abs pct  : {test_df['abs_pct_err'].mean():.2f}%")



## 3. Per-Store MAPE Breakdown

MAPE can vary significantly between stores. Isolating underperforming stores helps prioritise model improvements.


In [ ]:

# ── Per-Store MAPE ────────────────────────────────────────────────────────

if "store_id" in test_df.columns:
    store_mape = (
        test_df.groupby("store_id")
        .apply(lambda g: g["abs_pct_err"].mean())
        .reset_index()
        .rename(columns={0: "MAPE"})
        .sort_values("MAPE", ascending=False)
    )

    fig = px.bar(
        store_mape, x="store_id", y="MAPE",
        title="Per-Store MAPE — XGBoost Test Set",
        labels={"MAPE": "MAPE (%)", "store_id": "Store"},
        color="MAPE",
        color_continuous_scale="RdYlGn_r",
        template="plotly_white",
    )
    fig.add_hline(y=15, line_dash="dash", line_color="orange",
                  annotation_text="15% threshold", annotation_position="top right")
    fig.add_hline(y=10, line_dash="dash", line_color="green",
                  annotation_text="10% target", annotation_position="top right")
    fig.show()

    over_15 = store_mape[store_mape["MAPE"] > 15]
    print(f"Stores above 15% MAPE: {len(over_15)}")
    if not over_15.empty:
        print(over_15.to_string(index=False))
else:
    print("No store_id column present — per-store breakdown not applicable.")
    store_mape = pd.DataFrame()



## 4. Error Distribution


In [ ]:

# ── Error distribution histogram ──────────────────────────────────────────

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Absolute % Error Distribution",
                                    "Residual Distribution"))

fig.add_trace(go.Histogram(
    x=test_df["abs_pct_err"].dropna(),
    nbinsx=50, name="Abs % Error",
    marker_color="steelblue", opacity=0.8,
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=test_df["residual"],
    nbinsx=50, name="Residual",
    marker_color="mediumseagreen", opacity=0.8,
), row=1, col=2)

fig.update_xaxes(title_text="Abs % Error (%)", row=1, col=1)
fig.update_xaxes(title_text="Actual − Predicted", row=1, col=2)
fig.update_yaxes(title_text="Count")
fig.update_layout(
    title_text="Error Distributions — XGBoost Test Set",
    template="plotly_white", showlegend=False, height=400,
)
fig.show()

pct_under_10 = (test_df["abs_pct_err"].dropna() < 10).mean() * 100
pct_under_15 = (test_df["abs_pct_err"].dropna() < 15).mean() * 100
print(f"Predictions with abs error < 10%: {pct_under_10:.1f}%")
print(f"Predictions with abs error < 15%: {pct_under_15:.1f}%")



## 5. Confidence Interval Coverage

An 80% confidence interval should *contain* the actual value approximately 80% of the time. We test this using the XGBoost ±10% heuristic bounds (as defined in `generate_forecast`).


In [ ]:

# ── CI coverage ───────────────────────────────────────────────────────────

test_df["lower_bound"] = test_df["predicted"] * 0.9
test_df["upper_bound"] = test_df["predicted"] * 1.1
test_df["in_interval"] = (
    (test_df["sales_qty"] >= test_df["lower_bound"]) &
    (test_df["sales_qty"] <= test_df["upper_bound"])
)

coverage = test_df["in_interval"].mean() * 100
print(f"Confidence interval coverage (±10%): {coverage:.1f}%")
print(f"(Expected for a well-calibrated ±10% bound: ~80%)")

# Plot a sample of 120 days from the test set
sample = test_df.sort_values("date").groupby("date").mean(numeric_only=True).reset_index().head(120)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pd.concat([sample["date"], sample["date"].iloc[::-1]]),
    y=pd.concat([sample["upper_bound"], sample["lower_bound"].iloc[::-1]]),
    fill="toself", fillcolor="rgba(100,149,237,0.20)",
    line={"color":"rgba(255,255,255,0)"}, name="±10% Interval",
    hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=sample["date"], y=sample["sales_qty"],
    mode="markers+lines", name="Actual",
    marker={"color":"black","size":4}, line={"color":"black","width":1},
))
fig.add_trace(go.Scatter(
    x=sample["date"], y=sample["predicted"],
    mode="lines", name="Predicted",
    line={"color":"crimson","dash":"dash"},
))
fig.update_layout(
    title=f"CI Coverage = {coverage:.1f}%  (sample of test period, daily avg)",
    xaxis_title="Date", yaxis_title="Sales",
    template="plotly_white", hovermode="x unified",
)
fig.show()



## 6. Final Accuracy Report


In [ ]:

## Summary

| Analysis | What was measured |
|---|---|
| Walk-forward CV (3 folds) | Stability of MAPE across time windows |
| Residuals over time | Bias and systematic error patterns |
| Actual vs Predicted scatter | Outlier predictions |
| Per-store MAPE | Stores needing dedicated model tuning |
| Error distribution | Shape of prediction errors |
| CI coverage | Whether ±10% bounds are realistic |

**Rule of thumb**: MAPE < 10% = excellent, 10–15% = good, > 25% = model needs rework.
